# TOPTW with Mandatory Visits

Notebook 100 introduced TOPTW as an arc-flow ILP where every customer is **optional** — the solver picks a profitable subset that fits the per-vehicle budget `T_max`. Real fleets usually aren't that clean: a handful of stops are contracted or priority targets that **must** be served, while the rest stay opportunistic.

The natural encoding is one extra constraint on the visit indicator:

$$
y_i = 1 \quad \forall i \in \mathcal{M}
$$

where $\mathcal{M}$ is the set of mandatory customer indices. `solve_toptw_ilp` accepts this set via a new `mandatory=[...]` keyword.

This notebook walks through the trade-off: forcing low-value visits onto the schedule reduces total achievable profit, and tight time windows on a mandatory customer can turn a previously-feasible problem infeasible.

Bridge to satellite tasking: mandatory targets are contracted observations (paying customer, SLA-bound priority target), opportunistic ones are scientific bonus passes.

In [1]:
%load_ext autoreload
%autoreload 2

from vrp_lib import solve_toptw_ilp

## 1. Instance

Four customers placed at distance 100 from the depot in four directions. Each round-trip costs ~200 + service time, and `t_max = 625` admits any three of the four. Three customers carry profit 10; the fourth (node 4) carries profit 1 — so the natural drop, with no mandatory constraint, is node 4.

In [2]:
import math

def make_distance(points):
    n = len(points)
    m = [[0] * n for _ in range(n)]
    for i, (xi, yi) in enumerate(points):
        for j, (xj, yj) in enumerate(points):
            if i != j:
                m[i][j] = int(round(math.hypot(xi - xj, yi - yj)))
    return m

points  = [(0, 0), (100, 0), (0, 100), (-100, 0), (0, -100)]
profits = [0, 10, 10, 10, 1]
windows = [(0, 1000)] * 5
distance = make_distance(points)

T_MAX = 625
SERVICE = 5
HORIZON = 1000

for row in distance:
    print(row)

[0, 100, 100, 100, 100]
[100, 0, 141, 200, 141]
[100, 141, 0, 141, 200]
[100, 200, 141, 0, 141]
[100, 141, 200, 141, 0]


## 2. Baseline: every customer is optional

Without `mandatory`, the solver minimises waste by dropping the lowest-profit customer (node 4, profit 1). Three profit-10 visits remain → `total_profit = 30`.

In [3]:
baseline = solve_toptw_ilp(
    distance, profits, windows,
    num_vehicles=1,
    t_max=T_MAX,
    service_time=SERVICE,
    horizon=HORIZON,
    time_limit_seconds=10,
)

print(f"status         : {baseline.status}")
print(f"total_profit   : {baseline.total_profit}")
print(f"dropped        : {baseline.dropped}")
for k, route in enumerate(baseline.routes):
    print(f"vehicle {k}     : {route}")

status         : 1
total_profit   : 30
dropped        : [4]
vehicle 0     : [0, 1, 3, 2, 0]


## 3. Mandatory node 4

Add `mandatory=[4]`. The solver must now visit node 4 (profit 1) and can fit only two of the three profit-10 customers. Total profit drops from 30 to **21** (10 + 10 + 1).

In [4]:
forced = solve_toptw_ilp(
    distance, profits, windows,
    num_vehicles=1,
    t_max=T_MAX,
    service_time=SERVICE,
    horizon=HORIZON,
    mandatory=[4],
    time_limit_seconds=10,
)

print(f"status         : {forced.status}")
print(f"total_profit   : {forced.total_profit}")
print(f"dropped        : {forced.dropped}  (one of 1/2/3 — solver's choice)")
for k, route in enumerate(forced.routes):
    print(f"vehicle {k}     : {route}")

assert 4 not in forced.dropped, "mandatory customer must be served"
assert forced.total_profit == 21

status         : 1
total_profit   : 21
dropped        : [2]  (one of 1/2/3 — solver's choice)
vehicle 0     : [0, 3, 1, 4, 0]


## 4. Side-by-side

| | baseline | mandatory=[4] |
|---|---|---|
| total_profit | 30 | 21 |
| dropped | [4] (profit 1) | one of {1, 2, 3} (profit 10) |
| visits | 3 high-value | 2 high-value + 1 contracted |

The 9-point profit gap is the **cost of the SLA**: the price you pay for guaranteeing service on a low-value customer when the budget is tight. Useful for sizing the question "how much extra fleet capacity would I need to keep the SLA *and* the bonus visits?" — the same model with a larger `t_max` answers it.

## 5. Infeasibility when a mandatory window is unreachable

Mandatory + tight windows can over-constrain the problem. With node 2 at distance 200 but a window that closes at t=50, no schedule can serve it. Marking it mandatory turns "drop it and continue" into infeasibility — surfaced via `status != 1` and empty routes.

In [5]:
bad_points  = [(0, 0), (10, 0), (200, 0)]
bad_profits = [0, 5, 100]
bad_windows = [(0, 500), (0, 500), (0, 50)]   # node 2: window closes at 50, dist=200
bad_distance = make_distance(bad_points)

infeas = solve_toptw_ilp(
    bad_distance, bad_profits, bad_windows,
    num_vehicles=1,
    t_max=500,
    service_time=0,
    horizon=500,
    mandatory=[2],
    time_limit_seconds=10,
)

print(f"status         : {infeas.status}  (1 == success; anything else == no solution)")
print(f"routes         : {infeas.routes}")
print(f"dropped        : {infeas.dropped}")

status         : 3  (1 == success; anything else == no solution)
routes         : []
dropped        : []


## Takeaway

- One extra constraint, `y_i == 1` for each mandatory customer, converts TOPTW from a pure orienteering problem into a **mixed mandatory/optional** scheduler — the form most real fleets actually need.
- The cost shows up in two ways: total profit decreases (because high-value visits get displaced) and the feasibility region shrinks (mandatory + tight windows can be infeasible).
- For satellite tasking this is exactly the encoding for contracted observations alongside opportunistic targets — no new modelling primitive needed beyond what `solve_toptw_ilp` already gives you.